In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [129]:
df = pd.read_csv("../data/german_credit_data.csv")

### Pré processamento

In [130]:
# Tratamento dos valores ausentes

df["Saving accounts"] = df["Saving accounts"].fillna("No Saving Account")

df["Checking account"] = df["Checking account"].fillna("No Checking Account")

print("Valores ausentes após o tratamento:\n")
print(df.isnull().sum())

print("\nCategorias de Saving accounts:")
print(df["Saving accounts"].value_counts())

print("\nCategorias de Checking account:")
print(df["Checking account"].value_counts())

Valores ausentes após o tratamento:

Unnamed: 0          0
Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64

Categorias de Saving accounts:
Saving accounts
little               603
No Saving Account    183
moderate             103
quite rich            63
rich                  48
Name: count, dtype: int64

Categorias de Checking account:
Checking account
No Checking Account    394
little                 274
moderate               269
rich                    63
Name: count, dtype: int64


In [131]:
# 1. Separar X e y e converter a variável alvo 
X = df.drop('Risk', axis=1)
y = df['Risk'].map({'good': 0, 'bad': 1})

# 2. Divisão entre treino e teste 
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

# Tratamento das variáveis categóricas com OneHotEncoder
colunas_categoricas = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), colunas_categoricas)
    ],
    remainder='passthrough'
)


nomes_das_colunas = preprocessor.fit(X_train).get_feature_names_out()

X_train_final = pd.DataFrame(preprocessor.transform(X_train), columns=nomes_das_colunas, index=X_train.index)
X_test_final = pd.DataFrame(preprocessor.transform(X_test), columns=nomes_das_colunas, index=X_test.index)

# Exibindo os resultados e dimensões finais
print("Dimensões dos conjuntos:\n")
print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final : {X_test_final.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

print("\nPrimeiras linhas de X_train_final:")
display(X_train_final.head())

Dimensões dos conjuntos:

X_train_final: (800, 22)
X_test_final : (200, 22)
y_train: (800,)
y_test : (200,)

Primeiras linhas de X_train_final:


,cat__Sex_male,cat__Housing_own,cat__Housing_rent,cat__Saving accounts_little,cat__Saving accounts_moderate,cat__Saving accounts_quite rich,cat__Saving accounts_rich,cat__Checking account_little,cat__Checking account_moderate,cat__Checking account_rich,cat__Purpose_car,cat__Purpose_domestic appliances,cat__Purpose_education,cat__Purpose_furniture/equipment,cat__Purpose_radio/TV,cat__Purpose_repairs,cat__Purpose_vacation/others,remainder__Unnamed: 0,remainder__Age,remainder__Job,remainder__Credit amount,remainder__Duration
828,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,828.0,47.0,2.0,8335.0,36.0
997,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,997.0,38.0,2.0,804.0,12.0
148,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,148.0,28.0,2.0,5371.0,36.0
735,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,735.0,29.0,0.0,3990.0,36.0
130,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,130.0,24.0,2.0,8487.0,48.0


## Criando Modelo

In [132]:
RandomForestClassifier(random_state=42)

RandomForestClassifier(random_state=42)

In [133]:
# Modelo Baseline

rf = RandomForestClassifier(
    random_state=42
)

# Treinamento
rf.fit(X_train_final, y_train)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [134]:
# Classes previstas
y_pred = rf.predict(X_test_final)

# Probabilidade da classe positiva (Bad = 1)
y_prob = rf.predict_proba(X_test_final)[:, 1]

predicoes = pd.DataFrame({
    "Real": y_test.values,
    "Previsto": y_pred,
    "Probabilidade_Bad": y_prob
})

display(predicoes.head(10))

,Real,Previsto,Probabilidade_Bad
0,0,0,0.16
1,0,0,0.37
2,1,1,0.54
3,0,1,0.54
4,1,0,0.33
5,0,0,0.35
6,0,0,0.32
7,0,0,0.38
8,0,0,0.35
9,0,0,0.15


## Ajustes hiperparâmetros

In [135]:
rf_tuning = RandomForestClassifier(
    random_state=42
)
# Espaço de hiperparâmetros
param_grid = {

    "n_estimators": [100, 200, 300, 500],

    "max_depth": [None,5,10,15,20],

    "min_samples_split": [2,5,10],

    "min_samples_leaf": [1,2,5,10],

    "max_features": [
        "sqrt",
        "log2"
    ],

    "class_weight": [
        None,
        "balanced"
    ]

}
# Randomized Search
random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_grid,
    n_iter=50,
    scoring="recall",
    cv=5,
    random_state=42,
    n_jobs=-1
)
print("Configuração criada!")

Configuração criada!


In [136]:
random_search.fit(X_train_final, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'class_weight': [None, 'balanced'],
                                        'max_depth': [None, 5, 10, 15, 20],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 5, 10],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='recall')

In [137]:
best_rf = random_search.best_estimator_

# Predições
y_pred_tuned = best_rf.predict(X_test_final)

y_prob_tuned = best_rf.predict_proba(X_test_final)[:,1]

print("Modelo otimizado pronto!")

Modelo otimizado pronto!


In [138]:
# Métricas modelo otimizado
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

precision_tuned = precision_score(y_test, y_pred_tuned)

recall_tuned = recall_score(y_test, y_pred_tuned)

f1_tuned = f1_score(y_test, y_pred_tuned)

roc_auc_tuned = roc_auc_score(y_test, y_prob_tuned)

print("Modelo Otimizado\n")
print(f"Accuracy : {accuracy_tuned:.3f}")
print(f"Precision: {precision_tuned:.3f}")
print(f"Recall   : {recall_tuned:.3f}")
print(f"F1-score : {f1_tuned:.3f}")
print(f"ROC-AUC  : {roc_auc_tuned:.3f}")


print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        y_pred_tuned,
        target_names=["Good", "Bad"]
    )
)

Modelo Otimizado

Accuracy : 0.730
Precision: 0.541
Recall   : 0.667
F1-score : 0.597
ROC-AUC  : 0.775

Classification Report

              precision    recall  f1-score   support

        Good       0.84      0.76      0.80       140
         Bad       0.54      0.67      0.60        60

    accuracy                           0.73       200
   macro avg       0.69      0.71      0.70       200
weighted avg       0.75      0.73      0.74       200



### Salvando Modelo

In [139]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
os.makedirs("../processed", exist_ok=True)

joblib.dump(rf, "../models/random_forest_baseline.pkl")
joblib.dump(best_rf, "../models/random_forest_tuned.pkl")

joblib.dump(X_train_final, "../processed/X_train.pkl")
joblib.dump(X_test_final, "../processed/X_test.pkl")
joblib.dump(y_train, "../processed/y_train.pkl")
joblib.dump(y_test, "../processed/y_test.pkl")

print("Modelos e dados exportados com sucesso!")

Modelos e dados exportados com sucesso!


In [140]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    best_rf,
    "../models/random_forest_tuned.pkl"
)

['../models/random_forest_tuned.pkl']